# Original Gaussian
Original workflow: `old/GPN_ori.ipynb`. Imports now use the `amgpn` package.
Run from the repository root after an editable install. Configuration keys retain the original notebook/cell identifiers.


In [ ]:
from amgpn.config import require as _cfg_require, resolve as _cfg_resolve, format_value as _cfg_format
#GPN模型二次复现：更严谨的参数配置
from pathlib import Path
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
import numpy as np
from tqdm import tqdm
from amgpn.legacy.original.episodes import *
from amgpn.legacy.original.gaussian import GPN,GPNTrainer,count_parameters

def main():
    # 参数设置
    FEATURE_DIM = _cfg_require('old/GPN_ori.ipynb.cell0.main.FEATURE_DIM')
    N_WAY = _cfg_require('old/GPN_ori.ipynb.cell0.main.N_WAY')
    K_SHOT = _cfg_require('old/GPN_ori.ipynb.cell0.main.K_SHOT')
    Q_QUERY = _cfg_require('old/GPN_ori.ipynb.cell0.main.Q_QUERY')
    NUM_TASKS = _cfg_require('old/GPN_ori.ipynb.cell0.main.NUM_TASKS')
    EPOCHS = _cfg_require('old/GPN_ori.ipynb.cell0.main.EPOCHS')
    LR = _cfg_require('old/GPN_ori.ipynb.cell0.main.LR')
    BATCH_SIZE = _cfg_require('old/GPN_ori.ipynb.cell0.main.BATCH_SIZE')
    SEED = _cfg_require('old/GPN_ori.ipynb.cell0.main.SEED')
    
    # 数据路径
    # current_path=Path(__file__).resolve().parent
    dataset_path = _cfg_require('old/GPN_ori.ipynb.cell0.main.dataset_path')
    
    # 创建原始数据集
# Historical configuration omitted; provide values through private configuration.
# Historical configuration omitted; provide values through private configuration.
    train_dataset=UAVDataset(data_dir_path=dataset_path+_cfg_require('old/GPN_ori.ipynb.cell0.main.path'),seed=SEED,max_sample_count=_cfg_require('old/GPN_ori.ipynb.cell0.main.max_sample_count'))
    test_dataset=UAVDataset(data_dir_path=dataset_path+_cfg_require('old/GPN_ori.ipynb.cell0.main.path__2'),seed=SEED,max_sample_count=_cfg_require('old/GPN_ori.ipynb.cell0.main.max_sample_count__2'))
    
    # 创建元任务划分集
    meta_train_dataset = MetaDataset(train_dataset, 8, 1, 5,batch_size=BATCH_SIZE,num_tasks=NUM_TASKS,epochs=EPOCHS)
    meta_test_dataset = MetaDataset(test_dataset, N_WAY, K_SHOT, Q_QUERY,batch_size=BATCH_SIZE,num_tasks=_cfg_require('old/GPN_ori.ipynb.cell0.main.num_tasks'),epochs=_cfg_require('old/GPN_ori.ipynb.cell0.main.epochs'))
    
    # DataLoader
    train_loader = DataLoader(meta_train_dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True, num_workers=_cfg_require('old/GPN_ori.ipynb.cell0.main.num_workers'))
    test_loader = DataLoader(meta_test_dataset, batch_size=_cfg_require('old/GPN_ori.ipynb.cell0.main.batch_size'), shuffle=False)
    #创建模型
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    model_without_se = GPN(use_se=_cfg_require('old/GPN_ori.ipynb.cell0.main.use_se'))
    params_without_se = count_parameters(model_without_se)
    # 创建训练器
    meta_learner = GPNTrainer(model_without_se,device=device,feature_dim=FEATURE_DIM,lr=LR)    
    print("开始训练...")
    meta_learner.train(train_loader=train_loader,test_loader=test_loader,
                       num_tasks=NUM_TASKS,n_ways=N_WAY,epochs=EPOCHS,
                      save_path=_cfg_require('old/GPN_ori.ipynb.cell0.main.save_path'))
    
    print("训练完成！")

    meta_learner.evaluate(test_loader=test_loader,n_ways=N_WAY)

if __name__ == '__main__':
    main()

